# Исследование Research Outputs

## О swagger.json

In [65]:
# imports
import os
import json
import pandas as pd

Удалось найти `swagger.json`, который рендерится на странице API документации.

Своими средствами (redoc, swagger-ui) выполнить полноценный рендеринг всего контента не удалось - много сущностей, высокая вложенность объектов, и JS библиотеки с этим не справляются.

Поэтому посмотрим, что находится в эндпоинте отдельного `ResearchOutput`, т.е. какие атрибуты у него есть.

In [66]:
# path_to_spec = os.path.join(, '522-swagger-pretty.json')
def load_spec(path_to_pec) -> object:
    with open('notebooks/522-swagger-pretty.json', 'r') as f:
        return json.load(f)


In [67]:
api_spec = load_spec('notebooks/522-swagger-pretty.json')
api_spec['paths'][f'/research-outputs/{{id}}']['get']['responses']['200']['schema']

{'$ref': '#/definitions/WSResearchOutput'}

In [68]:
api_spec['paths'][f'/research-outputs']['get']['responses']['200']['schema']

{'$ref': '#/definitions/WSResearchOutputListResult'}

В данной спецификации есть такие `$ref`, которые указывают на схемы сущностей.
К счастью, они находятся внутри этого же файла.

Все схемы объявлены в `definitions`.

In [10]:
# get all attributes
research_output_schema = api_spec['definitions']['WSResearchOutput']['properties']

## Парсинг атрибутов сущности

Надо бы написать парсер...
У каждого атрибута есть 'type': это может быть как примитив, так и ссылка на другую сущность.
Тут приходится придумать какие-то эвристики на основе того, что встречается в выборке.

Если `type` это `string` или `boolean`, то этого достаточно.

Но у `string` м.б. еще и `format` (напр. `datetime`).

Если `type` это `integer`, то у него есть `format`, который указывает длину инта.

Если `type` это `number`, то у него есть `format`, в котором будет `double`.

Если `type` это `array`, то у него должны быть `items`

In [98]:
def fetch_schema(spec, schema_name):
    # pass schema name according to swagger spec
    # get nice pandas dataframe back

    schema_object = spec['definitions'][schema_name]['properties']
    parsed_schema = parse_schema(schema_object)

    return pd.DataFrame(parsed_schema).sort_values(by='field')

def parse_schema(schema):
    references = []
    
    for key, value in schema.items():
        row = { 'field': key }

        match value:
            case {'$ref': ref}:
                # just ref, i.e. link to another entity
                entity = ref.split('/')[-1]
                row['type'] = entity
            case {'type': 'array', 'items': { 'type': field_type }}:
                # likely this is a string then
                row['type'] = f'Array <Primitive<{field_type}>>'
            case {'type': 'array', 'items': { '$ref': ref }}:
                entity = ref.split('/')[-1]
                row['type'] = f'Array <{entity}>'
            case {'type': 'integer', 'format': int_format}:
                row['type'] = f'Primitive <{int_format}>'
            case {'type': 'number', 'format': number_format}:
                row['type'] = f'Primitive <{number_format}>'
            case {'type': 'string', 'format': string_format}:
                row['type'] = f'Primitive <string ({string_format})>'
            case {'type': 'string'}:
                row['type'] = f'Primitive <string>'
            case {'type': simple_type}:
                row['type'] = f'Primitive <{simple_type}>'
            case _:
                row['type'] = 'UNKNOWN'
        
        references.append(row)
    
    return references

In [99]:
research_schema_df = fetch_schema(api_spec, 'WSResearchOutput')
research_schema_df

,field,type
55,abstract,WSLocalizedString
29,additionalFiles,Array <WSElectronicVersionAdditionalFileAssoci...
30,additionalLinks,Array <WSLink>
34,articleProcessingChargeAmount,Primitive <double>
32,articleProcessingChargeAmountInArticleProcessi...,Primitive <double>
33,articleProcessingChargeCurrency,WSClassification
31,articleProcessingChargePaid,Primitive <boolean>
50,assessmentType,WSClassification
36,bibliographicalNote,WSLocalizedString
9,category,WSClassification


## Частота появления атрибутов

Поскольку нет информации о том, какие поля обязательно встретятся в записи о научной деятельности, посмотрим, как часто встречается каждый из найденных в спецификации атрибутов в тестовой выгрузке.

Это позволит обнаружить часто встречаемые атрибуты более надежно, нежели действовать из соображений здравого смысла.

### Загрузка данных

In [100]:
def load_data(path_to_data) -> object:
    with open(path_to_data, 'r') as f:
        return json.load(f)

In [ ]:
researches = load_data('../data/research-outputs.json')['items']
researches[0] # sanity check

### Подсчет частот

Исходя из схемы, представленной в спецификации, будем считать так: для каждого объекта выборки смотрим, какие ключи у него есть из схемы, инкрементируем, если нашли.

Но дополнительно будем вести учет атрибутов, которых в схеме нет, но по факту они встретились.

In [102]:
def count_field_occurences(items, keys):
    # items - list of objects to explore
    # keys - list of keys to search

    # returns two dicts: one counts occurences for keys that are found in schema
    #   the other one counts occurences for key that were not found in schema

    total_occurences = dict.fromkeys(keys, 0)
    exceptional_keys = dict()

    for item in items:
        for key in item.keys():
            if key in total_occurences.keys():
                total_occurences[key] += 1
            else:
                if key in exceptional_keys.keys():
                    exceptional_keys[key] += 1
                else:
                    exceptional_keys[key] = 1

    return total_occurences, exceptional_keys
    


In [103]:
occurences, unknown_fields = count_field_occurences(researches, list(research_df['field']))

In [ ]:
occurences_df = pd.DataFrame(occurences.items(), columns=['field', 'count'])
occurences_df.sort_values(by='count', ascending=False).info()

In [105]:
unknown_fields_df = pd.DataFrame(unknown_fields.items(), columns=['field', 'count'])
unknown_fields_df.sort_values(by='count', ascending=False)

,field,count
1,journalAssociation,969
0,pages,873
3,volume,340
2,journalNumber,244
4,number,49
7,hostPublicationTitle,31
5,publisher,28
9,isbns,25
6,event,16
8,articleNumber,10


In [106]:
result = result = pd.merge(research_df, occurences_df, on='field', how='outer')
result_sorted = result.sort_values(by='count', ascending=False)
# result_sorted

Посмотрим только на те атрибуты, которые есть для всех записей

In [107]:
sample_size = load_data('../data/research-outputs.json')['pageInformation']['size']
result_sorted[result_sorted['count'] == sample_size]

,field,type,count
9,category,WSClassification,1000
16,externalIdSource,Primitive <string>,1000
15,externalId,Primitive <string>,1000
10,confidential,Primitive <boolean>,1000
22,info,WSContentInformation,1000
29,openAccessPermission,WSClassification,1000
32,personAssociations,Array <WSClassifiedAuthorAssociation>,1000
27,managingOrganisationalUnit,WSOrganisationRef,1000
25,language,WSClassification,1000
48,title,WSValue,1000


Также можно посмотреть структуру каждой из схем

In [110]:
thing_schema = fetch_schema(api_spec, 'WSVisibility')
thing_schema

,field,type
0,key,Primitive <string>
1,value,WSLocalizedString


## Классификация

У каждой научной деятельность есть `category` и `type`.
Для них нет отдельных схем, они все `WSClassification`.

Судя по `classification-schemes.json`, оно все уместилось в тестовую выборку, и мы можем попробовать поработать с ними.

In [111]:
classification_schema = fetch_schema(api_spec, 'WSClassification')
classification_schema

,field,type
8,classificationRelations,Array <WSClassificationRelation>
7,description,WSLocalizedString
6,disabled,Primitive <boolean>
1,externalId,Primitive <string>
2,externalIdSource,Primitive <string>
3,externallyManaged,Primitive <boolean>
0,pureId,Primitive <int64>
5,term,WSLocalizedString
4,uri,Primitive <string>


Проведем такой же анализ и для `classification-schemes.json`.

In [112]:
classifications = load_data('../data/classification-schemes.json')['items']
occurences, unknown_fields = count_field_occurences(classifications, list(classification_schema['field']))

In [122]:
occurences_df = pd.DataFrame(occurences.items(), columns=['field', 'count'])
result = result = pd.merge(classification_schema, occurences_df, on='field', how='outer')
result_sorted = result.sort_values(by='count', ascending=False)
result_sorted

,field,type,count
6,pureId,Primitive <int64>,315
1,description,WSLocalizedString,312
0,classificationRelations,Array <WSClassificationRelation>,0
3,externalId,Primitive <string>,0
2,disabled,Primitive <boolean>,0
4,externalIdSource,Primitive <string>,0
5,externallyManaged,Primitive <boolean>,0
7,term,WSLocalizedString,0
8,uri,Primitive <string>,0


In [117]:
unknown_fields_df = pd.DataFrame(unknown_fields.items(), columns=['field', 'count'])
unknown_fields_df.sort_values(by='count', ascending=False)

,field,count
0,uuid,315
1,baseUri,315
2,info,315
4,containedClassifications,281
3,typeClassification,264
5,associatedSchemes,1


## Связь с организациями и людьми

Имеет смысл понять, каким образом связаны между собой исследования, люди и организации

### Как выяснить авторов?..

`WSResearchOutput` содержит `personAssociations`, который - массив `WSClassifiedAuthorAssociation`.
То есть у одного ресерча может быть несколько авторов.

Посмотрим на структуру `WSClassifiedAuthorAssociation`.

In [ ]:
# get person associations
classified_author_schema = data['definitions']['WSClassifiedAuthorAssociation']['properties']
parsed_cl_author_schema = parse_schema(classified_author_schema)
cl_author_df = pd.DataFrame(parsed_cl_author_schema).sort_values(by='field')
cl_author_df

Кроме прочих метаданных, `WSClassifiedAuthorAssociation` содержит ссылку на человека `WSPersonRef`.
Посмотрим, какие атрибуты есть у него:

In [ ]:
person_ref_schema = data['definitions']['WSPersonRef']['properties']
parsed_person_ref_schema = parse_schema(person_ref_schema)
person_ref_df = pd.DataFrame(parsed_person_ref_schema).sort_values(by='field')
person_ref_df

### Как выяснить организации?

Похожую операцию можем повторить и для организаций

In [ ]:
org_unit_schema = data['definitions']['WSOrganisationRef']['properties']
parsed_org_unit_schema = parse_schema(org_unit_schema)
org_unit_df = pd.DataFrame(parsed_org_unit_schema).sort_values(by='field')
org_unit_df

## Какие поля важны?

Не все поля нам нужны для какого-то первичного анализа.
К тому же все равно неясно, какие атрибуты будут присутствовать у каждой записи.

Поэтому, опираясь на свои представления о домене, отберем набор атрибутов, которые наверняка будут встречаться везде.

К таким можно отнести:

- `uuid` (*)
- `title` (*)
- `submissionYear` (нету)
- `type` (*)
- `category` (*)
- `managingOrganisationalUnit` (*)
- `personAssociations` (*)
- `electronicVersions`

Более сложные, не на сейчас:

- `keywordGroups`

Посмотрим, как представлены `type` и `category`

In [ ]:
# get classification
classification_schema = data['definitions']['WSClassification']['properties']
parsed_classification_schema = parse_schema(classification_schema)
classification_df = pd.DataFrame(parsed_classification_schema).sort_values(by='field')
classification_df

In [ ]:
def load_data(path_to_data) -> object:
    with open(path_to_data, 'r') as f:
        return json.load(f)

In [ ]:
researches = load_data('../data/research-outputs.json')['items']
researches[0] # sanity check

### Получение интересующих значений

Вытащим обозначенные выше атрибуты в dataframe и посмотрим на результат

In [ ]:
def get_electronic_version(obj):
    if 'electronicVersions' not in obj:
        return []
    
    result = []

    for version in obj['electronicVersions']:
        if 'doi' in version:
            result.append(version['doi'])
        if 'link' in version:
            result.append(version['link'])
    
    return result

def get_person_uuids(persons):
    result = []
    for person in persons:
        if 'person' in person:
            result.append(person['person']['uuid'])
        elif 'externalPerson' in person:
            result.append(f'Ext <{person['externalPerson']['uuid']}>')
    
    return result

def read_researches(research_list):
    result = []

    for research in research_list:
        research_filtered = {
            'uuid': research['uuid'],
            'title': research['title']['value'],
            'type': research['type']['pureId'],
            'category': research['category']['pureId'],
            'managing_org_unit': research['managingOrganisationalUnit']['uuid'],
            'contributors': get_person_uuids(research['personAssociations']),
            'electronicVersions': get_electronic_version(research)
        }

        result.append(research_filtered)
    
    return result

### Результат

In [ ]:
filtered_researches = read_researches(researches)
researches_df = pd.DataFrame(filtered_researches)

Посмотрим, сколько исследований из выгрузки имеют хоть какие-то ссылки на сторонние источники

In [224]:
researches_df[researches_df['electronicVersions'].str.len() > 0]

,uuid,title,type,category,managing_org_unit,contributors,electronicVersions
5,9cd7de97-4eac-4f80-ae14-1d735c50113c,ЛЬВОВА Д.А. ПРОФЕССИОНАЛЬНЫЕ ОБЪЕДИНЕНИЯ БУХГА...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[5b9b301a-69c4-4bf1-b493-6fdc9f7ca918],[http://elibrary.ru/item.asp?id=11632741]
6,74bf478c-2d88-4aea-909d-3efcc6ae032b,"РЕЦЕНЗИЯ НА МОНОГРАФИЮ В.Ф. СЫЧА ""МОРФОЛОГИЯ Л...",4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[aed55111-b3f4-4cab-9542-aa5864aad36c],[http://elibrary.ru/item.asp?id=11845914]
8,34ab6333-82e9-4030-b824-cdf9c459d39d,ГОЛОВИН Н. А. ТЕОРЕТИКО-МЕТОДОЛОГИЧЕСКИЕ ОСНОВ...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[1c58e000-f429-4c46-bc79-efab3cc369b3],[http://elibrary.ru/item.asp?id=12867462]
9,9a4deab5-47a6-4f81-9209-12c88d44beaa,DAVIS R. TYPING POLITICS: THE ROLE OF BLOGS IN...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[cafe0010-4f6a-40c8-846f-cb2568c4686f],[http://elibrary.ru/item.asp?id=16973104]
12,37598330-9615-4e55-83e9-dc240c8adb72,МИФ ЦИФРОВОЙ ДЕМОКРАТИИ. РЕЦЕНЗИЯ НА КНИГУ: HI...,4082,3943,09971b19-384c-4b78-b17c-6ea2921a31a7,[cafe0010-4f6a-40c8-846f-cb2568c4686f],[http://elibrary.ru/item.asp?id=16401209]
...,...,...,...,...,...,...,...
995,454dc5a2-d20a-40f7-b83b-bf4b0f3ead71,One-dimensional model of a distributed conductor,3973,3937,09971b19-384c-4b78-b17c-6ea2921a31a7,"[Ext <a4cd8abc-0032-4b29-a421-a03952a43e2e>, 9...",[http://proxy.library.spbu.ru:2095/article/10....
996,ac93fe35-a752-4578-a6b7-0d884d819397,Potential splitting approach to the three-body...,3973,3937,2ee32a65-7c0d-4432-94d0-6f419754d2de,"[Ext <d7079f7f-3fff-4d68-beec-795c07b6e48b>, f...",[https://doi.org/10.1209/0295-5075/110/30006]
997,03f7fffc-608b-4d89-8e13-8f9521e17762,Mathematical model and software complex for co...,4029,3937,09971b19-384c-4b78-b17c-6ea2921a31a7,[49da2164-3749-4c69-91b3-f43c040ec477],"[https://doi.org/10.1063/1.4912673, http://sci..."
998,586ceb2f-89bb-4afd-ae50-5c4c99c97115,Internal impedance of steel-reinforced helical...,3973,3937,09971b19-384c-4b78-b17c-6ea2921a31a7,"[Ext <a4cd8abc-0032-4b29-a421-a03952a43e2e>, 9...","[https://doi.org/10.1134/S1063785015040288, ht..."


Среди них есть ссылки с doi; уже что-то, что можно использовать с какими-то внешними инструментами